In [1]:
from typing import Any

import pandas as pd
from pathlib import Path

from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers.util import semantic_search

load_dotenv()

C:\2026-Projects\Document_Intelligent_Hub\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
NOTEBOOK_DIR =Path.cwd()

DATA_DIR = NOTEBOOK_DIR / "data"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"/ "chroma"

print(f"Current working directory: {NOTEBOOK_DIR}")
print(f"Vector store directory: {VECTOR_STORE_DIR}")
print(f"Vector store exists: {VECTOR_STORE_DIR.exists()}")

Current working directory: C:\2026-Projects\Document_Intelligent_Hub\notebooks
Vector store directory: C:\2026-Projects\Document_Intelligent_Hub\notebooks\data\vector_store\chroma
Vector store exists: True


In [3]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_function = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4646.60it/s]


In [4]:
COLLECTION_NAME = "enterprise_documents"

vector_store = Chroma(
    persist_directory=str(VECTOR_STORE_DIR),
    embedding_function=embedding_function,
    collection_name=COLLECTION_NAME,
)

collection_count = vector_store._collection.count()


print(f"Loaded Chroma collection: {COLLECTION_NAME}")
print(f"Documents/chunks in collection: {collection_count}")

Loaded Chroma collection: enterprise_documents
Documents/chunks in collection: 5


In [5]:
query = "Which AI project is best for banks and government?"

results = vector_store.similarity_search(
    query=query,
    k=5,
)

print(f"Query: {query}")
print(f"Results: {len(results)}")

Query: Which AI project is best for banks and government?
Results: 5


In [6]:
for index, document in enumerate(results, start=1):
    print("=" * 100)
    print(f"Result {index}")
    print("=" * 100)
    print("Source:", document.metadata.get("source"))
    print("Page:", document.metadata.get("page_number"))
    print("Document ID:", document.metadata.get("document_id"))
    print("Chunk ID:", document.metadata.get("chunk_id"))
    print()
    print(document.page_content[:1000])

Result 1
Source: Sample Compliance Manual.pdf
Page: 1
Document ID: 9c1bdd10-0455-4093-b96e-f21ccd2131df
Chunk ID: 40b1525f-5997-4905-993a-07e5013123b1

• Transactions involving foreign accounts must be flagged for enhanced due 
diligence. 
Suspicious Activity Indicators 
Employees must be alert to: 
• Structuring deposits to avoid reporting thresholds. 
• Rapid movement of funds between unrelated accounts. 
• Use of shell companies with unclear ownership. 
• Transactions inconsistent with customer profiles. 
Training & Certification 
• AML training is mandatory annually for all employees. 
• Certification records must be retained for 5 years.
Result 2
Source: Sample Compliance Manual.pdf
Page: 1
Document ID: 9c1bdd10-0455-4093-b96e-f21ccd2131df
Chunk ID: 2e963208-afd2-4d35-b36c-0c92fe541fbc

Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management acro

In [7]:
query = "what should I do if I am not happy with the AI project?"

scored_results= vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)
for index, (document, score) in enumerate(scored_results, start=1):
    print("=" * 100)
    print(f"Result {index}")
    print(f"Score: {score:.4f}")
    print("=" * 100)
    print("Source:", document.metadata.get("source"))
    print("Page:", document.metadata.get("page_number"))
    print()
    print(document.page_content[:1000])

Result 1
Score: -0.1956
Source: Sample Compliance Manual.pdf
Page: 2

• High-risk findings must be escalated to the Risk Committee within 7 business 
days. 
Business Continuity 
• Continuity plans must be tested annually. 
• Disaster recovery drills must be documented and reviewed. 
Vendor Compliance 
• Vendors must provide proof of compliance with ISO 27001 standards.
Result 2
Score: -0.3484
Source: Sample Compliance Manual.pdf
Page: 1

• Transactions involving foreign accounts must be flagged for enhanced due 
diligence. 
Suspicious Activity Indicators 
Employees must be alert to: 
• Structuring deposits to avoid reporting thresholds. 
• Rapid movement of funds between unrelated accounts. 
• Use of shell companies with unclear ownership. 
• Transactions inconsistent with customer profiles. 
Training & Certification 
• AML training is mandatory annually for all employees. 
• Certification records must be retained for 5 years.
Result 3
Score: -0.4215
Source: Sample Compliance Manual.pd

C:\Users\benne\AppData\Local\Temp\ipykernel_3376\165105079.py:3: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Sample Compliance Manual.pdf_3', metadata={'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'creator': 'Microsoft® Word for Microsoft 365', 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'chunk_index': 3, 'source': 'Sample Compliance Manual.pdf', 'file_type': 'pdf', 'chunk_size': 301, 'producer': 'Microsoft® Word for Microsoft 365', 'author': 'BENNET DYANI', 'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'page_label': '2', 'chunk_id': 'ccdd3435-52ed-4582-8713-cf0a7381e819', 'creationdate': '2026-06-20T21:35:22+02:00', 'part_index': 1, 'access_role': 'Compliance Analyst', 'department': 'Compliance', 'moddate': '2026-06-20T21:35:22+02:00', 'page_number': 2, 'page': 1, 'total_pages': 3}, page_content='• High-risk findings must be escalated to the Risk Committee within 7 business

In [8]:
def search_results_to_dataframe(
        results: list[tuple[Document, float | int]],
) -> pd.DataFrame:
    rows = []

    for rank, result in enumerate(results, start=1):
        document, score = result

        rows.append(
            {
                "rank": rank,
                "score": round(score, 4),
                "source": document.metadata.get("source"),
                "page_number": document.metadata.get("page_number"),
                "department": document.metadata.get("department"),
                "access_role": document.metadata.get("access_role"),
                "chunk_id": document.metadata.get("chunk_id"),
                "preview": document.page_content[:300],
            }
        )

        return pd.DataFrame(rows)

In [9]:
query = "secure knowledge management for enterprises policies"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)

results_df = search_results_to_dataframe(scored_results)

results_df

C:\Users\benne\AppData\Local\Temp\ipykernel_3376\196221103.py:3: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Sample Compliance Manual.pdf_2', metadata={'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'page': 1, 'file_type': 'pdf', 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'creator': 'Microsoft® Word for Microsoft 365', 'page_number': 2, 'total_pages': 3, 'access_role': 'Compliance Analyst', 'part_index': 1, 'chunk_size': 955, 'department': 'Compliance', 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'source': 'Sample Compliance Manual.pdf', 'moddate': '2026-06-20T21:35:22+02:00', 'creationdate': '2026-06-20T21:35:22+02:00', 'page_label': '2', 'chunk_index': 2, 'author': 'BENNET DYANI', 'producer': 'Microsoft® Word for Microsoft 365', 'chunk_id': '9ff8d3c6-7ab6-499f-911f-984b2b7d4000'}, page_content='Page 3 — Data Privacy & Protection \nData Storage \n• Personal data must be st

,rank,score,source,page_number,department,access_role,chunk_id,preview
0,1,0.1583,Sample Compliance Manual.pdf,2,Compliance,Compliance Analyst,9ff8d3c6-7ab6-499f-911f-984b2b7d4000,Page 3 — Data Privacy & Protection \nData Stor...


In [10]:
def semantic_search(
        query: str,
        k: int = 5,
) -> list[dict[str, Any]]:
    scored_results = vector_store.similarity_search_with_relevance_scores(
        query=query,
        k=5,
    )

    formatted_results =[]

    for rank, result in enumerate(scored_results, start=1):
        document, score = result

        formatted_results.append(
            {
                "rank": rank,
                "score": float(score),
                "content": document.page_content,
                "preview": document.page_content[:500],
                "metadata": document.metadata,
                "source": document.metadata.get("source"),
                "page_number": document.metadata.get("page_number"),
                "chunk_id": document.metadata.get("chunk_id"),
                "department": document.metadata.get("department"),
                "access_role": document.metadata.get("access_role"),
            }
        )

        return formatted_results

In [11]:
def format_result_for_ui(result: dict[str, Any]) -> dict[str, Any]:
    metadata = result["metadata"]
    return{
        "title": metadata.get("title") or result.get("source") or "Unknown Document",
        "source": result.get("source") or "Unknown Source",
        "page": result.get("page_number") or "N/A",
        "score": round(result.get("score", 0), 4),
        "preview": result.get("preview", ""),
        "department": result.get("department") or "Unknown",
        "access_role": result.get("access_role") or "Unknown",
        "chunk_id": result.get("chunk_id"),
    }

results = semantic_search(
    query="compliance document intelligence hub",
    k=5,
)

for result in results:
    print("=" * 80)
    print(f"Rank: {result['rank']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page_number']}")
    print()
    print(result["preview"])

Rank: 1
Score: 0.1439
Source: Sample Compliance Manual.pdf
Page: 1

Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, and third-party vendors engaged with the organization. 
Compliance is not optional. Failure to adhere to these standards may result in 
disciplinary action, financial penalties, or regulatory sanctions. 
The manual is designed to: 
• Provide


C:\Users\benne\AppData\Local\Temp\ipykernel_3376\3109272155.py:5: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Sample Compliance Manual.pdf_0', metadata={'moddate': '2026-06-20T21:35:22+02:00', 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'chunk_size': 991, 'part_index': 0, 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'chunk_id': '2e963208-afd2-4d35-b36c-0c92fe541fbc', 'creator': 'Microsoft® Word for Microsoft 365', 'chunk_index': 0, 'page': 0, 'producer': 'Microsoft® Word for Microsoft 365', 'page_number': 1, 'author': 'BENNET DYANI', 'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'page_label': '1', 'file_type': 'pdf', 'department': 'Compliance', 'creationdate': '2026-06-20T21:35:22+02:00', 'source': 'Sample Compliance Manual.pdf', 'access_role': 'Compliance Analyst', 'total_pages': 3}, page_content='Sample Compliance Manual \n \nPage 1 — Introduction & Scope \nThis Compliance

In [12]:
ui_results = [format_result_for_ui(result) for result in results]

pd.DataFrame(ui_results)

,title,source,page,score,preview,department,access_role,chunk_id
0,Sample Compliance Manual.pdf,Sample Compliance Manual.pdf,1,0.1439,Sample Compliance Manual \n \nPage 1 — Introdu...,Compliance,Compliance Analyst,2e963208-afd2-4d35-b36c-0c92fe541fbc


In [13]:
test_queries = [
    "best enterprise AI project for banks",
    "compliance and policy document intelligence",
    "workflow orchestrator for reports and audits",
    "secure knowledge management with citations",
    "what should I build first to maximize hireability?",
    "which projects are useful for government entities?",
]

for query in test_queries:
    print("=" * 120)
    print(f"QUERY: {query}")
    print("=" * 120)

    results = semantic_search(query=query, k=3)

    for result in results:
        print(f"\nRank: {result['rank']} | Score: {result['score']:.4f} |")
        print(f"Source: {result['source']} | Page: {result['page_number']}")
        print(result["preview"][:400])

QUERY: best enterprise AI project for banks

Rank: 1 | Score: -0.0203 |
Source: Sample Compliance Manual.pdf | Page: 1
• Transactions involving foreign accounts must be flagged for enhanced due 
diligence. 
Suspicious Activity Indicators 
Employees must be alert to: 
• Structuring deposits to avoid reporting thresholds. 
• Rapid movement of funds between unrelated accounts. 
• Use of shell companies with unclear ownership. 
• Transactions inconsistent with customer profiles. 
Training & Certification 
• AML traini
QUERY: compliance and policy document intelligence

Rank: 1 | Score: 0.2191 |
Source: Sample Compliance Manual.pdf | Page: 1
Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, and third-party vendors engaged with the organization. 
Compliance is not optional. Failure to a

C:\Users\benne\AppData\Local\Temp\ipykernel_3376\3109272155.py:5: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Sample Compliance Manual.pdf_1', metadata={'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'chunk_size': 498, 'producer': 'Microsoft® Word for Microsoft 365', 'total_pages': 3, 'page': 0, 'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'source': 'Sample Compliance Manual.pdf', 'moddate': '2026-06-20T21:35:22+02:00', 'department': 'Compliance', 'chunk_index': 1, 'access_role': 'Compliance Analyst', 'file_type': 'pdf', 'creator': 'Microsoft® Word for Microsoft 365', 'part_index': 0, 'creationdate': '2026-06-20T21:35:22+02:00', 'page_number': 1, 'page_label': '1', 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'chunk_id': '40b1525f-5997-4905-993a-07e5013123b1', 'author': 'BENNET DYANI'}, page_content='• Transactions involving foreign accounts must be flagged for enhanced due \n

In [14]:
filtered_results = vector_store.similarity_search_with_relevance_scores(
    query = "compliance risk flagging",
    k = 5,
    filter={
        "department": "compliance",
    },
)

search_results_to_dataframe(filtered_results)

In [15]:
pdf_results = vector_store.similarity_search_with_relevance_scores(
    query = "document intelligence hub",
    k = 5,
    filter={
        "file_type": "pdf",
    },

)

search_results_to_dataframe(pdf_results)

C:\Users\benne\AppData\Local\Temp\ipykernel_3376\3333771168.py:1: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Sample Compliance Manual.pdf_0', metadata={'author': 'BENNET DYANI', 'chunk_size': 991, 'total_pages': 3, 'part_index': 0, 'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'file_type': 'pdf', 'chunk_index': 0, 'creationdate': '2026-06-20T21:35:22+02:00', 'department': 'Compliance', 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'access_role': 'Compliance Analyst', 'source': 'Sample Compliance Manual.pdf', 'creator': 'Microsoft® Word for Microsoft 365', 'page': 0, 'chunk_id': '2e963208-afd2-4d35-b36c-0c92fe541fbc', 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'page_label': '1', 'moddate': '2026-06-20T21:35:22+02:00', 'page_number': 1, 'producer': 'Microsoft® Word for Microsoft 365'}, page_content='Sample Compliance Manual \n \nPage 1 — Introduction & Scope \nThis Compliance

,rank,score,source,page_number,department,access_role,chunk_id,preview
0,1,-0.2099,Sample Compliance Manual.pdf,1,Compliance,Compliance Analyst,2e963208-afd2-4d35-b36c-0c92fe541fbc,Sample Compliance Manual \n \nPage 1 — Introdu...


In [16]:
def semantic_search_with_rbac(
        query: str,
        user_role: str,
        k: int = 5,
) -> list[dict[str, Any]]:
    scored_results = vector_store.similarity_search_with_relevance_scores(
        query=query,
        k=k,
        filter={
            "access_role": user_role,
        },
    )

    formatted_results = []

    for rank, result in enumerate(scored_results, start=1):
        document, score = result

        formatted_results.append(
            {
                "rank": rank,
                "score": float(score),
                "content": document.page_content,
                "preview": document.page_content[:500],
                "metadata": document.metadata,
                "source": document.metadata.get("source"),
                "page_number": document.metadata.get("page_number"),
                "chunk_id": document.metadata.get("chunk_id"),
                "department": document.metadata.get("department"),
                "access_role": document.metadata.get("access_role"),
            }
        )

    return formatted_results

In [17]:
rbac_results = semantic_search_with_rbac(
    query="compliance document intelligence hub",
    user_role="compliance_analyst",
    k=5,
)

pd.DataFrame([format_result_for_ui(result) for result in rbac_results])

""


In [18]:
def normalize_user_role(user_role: str) -> str:
    role = user_role.strip().lower().replace("-", "_").replace(" ", "_")

    role_map = {
        "admin": "Admin",
        "compliance_analyst": "Compliance Analyst",
        "viewer": "Viewer",
    }

    return role_map.get(role, user_role)

In [19]:
def semantic_search_with_role_access(
    query: str,
    user_role: str,
    k: int = 5,
) -> list[dict]:
    if user_role == "Admin":
        scored_results = vector_store.similarity_search_with_relevance_scores(
            query=query,
            k=k,
        )
    else:
        scored_results = vector_store.similarity_search_with_relevance_scores(
            query=query,
            k=k,
            filter={
                "access_role": user_role,
            },
        )

    formatted_results = []

    for rank, result in enumerate(scored_results, start=1):
        document, score = result

        formatted_results.append(
            {
                "rank": rank,
                "score": float(score),
                "content": document.page_content,
                "preview": document.page_content[:500],
                "metadata": document.metadata,
                "source": document.metadata.get("source"),
                "page_number": document.metadata.get("page_number"),
                "chunk_id": document.metadata.get("chunk_id"),
                "department": document.metadata.get("department"),
                "access_role": document.metadata.get("access_role"),
            }
        )

    return formatted_results

In [20]:
admin_results = semantic_search_with_role_access(
    query="best system for government and finance",
    user_role="Admin",
    k=5,
)

pd.DataFrame([format_result_for_ui(result) for result in admin_results])

compliance_results = semantic_search_with_rbac(
    query="best system for government and finance",
    user_role="compliance_analyst",
    k=5,
)

pd.DataFrame([format_result_for_ui(result) for result in compliance_results])

C:\Users\benne\AppData\Local\Temp\ipykernel_3376\4231444893.py:7: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Sample Compliance Manual.pdf_1', metadata={'author': 'BENNET DYANI', 'page_number': 1, 'creator': 'Microsoft® Word for Microsoft 365', 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'moddate': '2026-06-20T21:35:22+02:00', 'chunk_id': '40b1525f-5997-4905-993a-07e5013123b1', 'chunk_index': 1, 'page_label': '1', 'part_index': 0, 'access_role': 'Compliance Analyst', 'file_type': 'pdf', 'source': 'Sample Compliance Manual.pdf', 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'chunk_size': 498, 'department': 'Compliance', 'total_pages': 3, 'creationdate': '2026-06-20T21:35:22+02:00', 'page': 0, 'producer': 'Microsoft® Word for Microsoft 365', 'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf'}, page_content='• Transactions involving foreign accounts must be flagged for enhanced due \n

""


In [21]:
def display_search_results(results: list[dict[str, Any]]) -> None:
    if not results:
        print("No results found.")
        return

    for result in results:
        print("=" * 100)
        print(f"Rank: {result['rank']}")
        print(f"Score: {result['score']:.4f}")
        print(f"Source: {result['source']}")
        print(f"Page: {result['page_number']}")
        print(f"Department: {result['department']}")
        print(f"Access Role: {result['access_role']}")
        print("-" * 100)
        print(result["preview"])
        print()

results = semantic_search_with_role_access(
    query="compliance document intelligence for banks",
    user_role="Compliance Analyst",
    k=5,
)

display_search_results(results)

Rank: 1
Score: 0.3325
Source: Sample Compliance Manual.pdf
Page: 1
Department: Compliance
Access Role: Compliance Analyst
----------------------------------------------------------------------------------------------------
Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, and third-party vendors engaged with the organization. 
Compliance is not optional. Failure to adhere to these standards may result in 
disciplinary action, financial penalties, or regulatory sanctions. 
The manual is designed to: 
• Provide

Rank: 2
Score: 0.2628
Source: Sample Compliance Manual.pdf
Page: 1
Department: Compliance
Access Role: Compliance Analyst
----------------------------------------------------------------------------------------------------
• Transactions involving foreign accounts must be fl

In [22]:
results = semantic_search_with_role_access(
    query="AML reporting requirements",
    user_role="Compliance Analyst",
    k=5,
)

print(len(results))
display_search_results(results)

5
Rank: 1
Score: 0.3919
Source: Sample Compliance Manual.pdf
Page: 1
Department: Compliance
Access Role: Compliance Analyst
----------------------------------------------------------------------------------------------------
Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, and third-party vendors engaged with the organization. 
Compliance is not optional. Failure to adhere to these standards may result in 
disciplinary action, financial penalties, or regulatory sanctions. 
The manual is designed to: 
• Provide

Rank: 2
Score: 0.3638
Source: Sample Compliance Manual.pdf
Page: 1
Department: Compliance
Access Role: Compliance Analyst
----------------------------------------------------------------------------------------------------
• Transactions involving foreign accounts must be 

C:\Users\benne\AppData\Local\Temp\ipykernel_3376\4231444893.py:12: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Sample Compliance Manual.pdf_0', metadata={'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'creationdate': '2026-06-20T21:35:22+02:00', 'page_label': '1', 'page_number': 1, 'author': 'BENNET DYANI', 'access_role': 'Compliance Analyst', 'department': 'Compliance', 'chunk_id': '2e963208-afd2-4d35-b36c-0c92fe541fbc', 'chunk_index': 0, 'moddate': '2026-06-20T21:35:22+02:00', 'total_pages': 3, 'part_index': 0, 'creator': 'Microsoft® Word for Microsoft 365', 'source': 'Sample Compliance Manual.pdf', 'file_type': 'pdf', 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'chunk_size': 991, 'page': 0, 'producer': 'Microsoft® Word for Microsoft 365'}, page_content='Sample Compliance Manual \n \nPage 1 — Introduction & Scope \nThis Complianc

In [23]:
results = semantic_search_with_role_access(
    query="AML reporting requirements",
    user_role="Admin",
    k=5,
)

print(len(results))
display_search_results(results)

5
Rank: 1
Score: 0.3919
Source: Sample Compliance Manual.pdf
Page: 1
Department: Compliance
Access Role: Compliance Analyst
----------------------------------------------------------------------------------------------------
Sample Compliance Manual 
 
Page 1 — Introduction & Scope 
This Compliance Manual sets forth the standards governing Anti-Money Laundering 
(AML), Data Privacy, and Operational Risk Management across the enterprise. It applies 
to all employees, contractors, and third-party vendors engaged with the organization. 
Compliance is not optional. Failure to adhere to these standards may result in 
disciplinary action, financial penalties, or regulatory sanctions. 
The manual is designed to: 
• Provide

Rank: 2
Score: 0.3638
Source: Sample Compliance Manual.pdf
Page: 1
Department: Compliance
Access Role: Compliance Analyst
----------------------------------------------------------------------------------------------------
• Transactions involving foreign accounts must be 

C:\Users\benne\AppData\Local\Temp\ipykernel_3376\4231444893.py:7: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Sample Compliance Manual.pdf_0', metadata={'file_path': 'C:\\2026-Projects\\Document_Intelligent_Hub\\notebooks\\data\\raw\\Sample Compliance Manual.pdf', 'moddate': '2026-06-20T21:35:22+02:00', 'author': 'BENNET DYANI', 'page': 0, 'page_number': 1, 'source': 'Sample Compliance Manual.pdf', 'producer': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-06-20T21:35:22+02:00', 'part_index': 0, 'ingested_at': '2026-06-20T19:37:26.921072+00:00', 'file_type': 'pdf', 'department': 'Compliance', 'chunk_id': '2e963208-afd2-4d35-b36c-0c92fe541fbc', 'document_id': '9c1bdd10-0455-4093-b96e-f21ccd2131df', 'access_role': 'Compliance Analyst', 'page_label': '1', 'chunk_size': 991, 'total_pages': 3, 'creator': 'Microsoft® Word for Microsoft 365', 'chunk_index': 0}, page_content='Sample Compliance Manual \n \nPage 1 — Introduction & Scope \nThis Compliance